# Module 10 - 실습 1: 토큰 예산 관리 (TokenBudget)

## 학습 목표
- `tiktoken`으로 텍스트의 정확한 토큰 수를 계산할 수 있다
- 문자 수와 토큰 수의 차이를 이해한다
- `TokenBudget` 클래스로 LLM 프롬프트 토큰 예산을 관리할 수 있다

## 참조 자료
- 학습 문서: `docs/part4-production/10-resource-optimization.md` (섹션 1.2, 2.4)

## 개념 요약

**문자 수 vs 토큰 수**:
```
영어:  "Hello world"  = 11 문자 = 2 토큰
한글:  "안녕하세요"    =  5 문자 = 5~7 토큰
코드:  "def hello():" = 13 문자 = 4 토큰
```

**TokenBudget 사용 패턴**:
```python
budget = TokenBudget(max_tokens=8000)
budget.consume(system_prompt)           # 고정 텍스트 소비
budget.consume(user_query)              # 고정 텍스트 소비
code = budget.truncate_to_budget(code)  # 남은 예산에 맞춰 절삭
```

In [ ]:
import sys
sys.path.insert(0, '..')

try:
    import tiktoken
    print("tiktoken 설치 확인 완료")
except ImportError:
    print("tiktoken을 설치해주세요: pip install tiktoken")

## 실습 1: tiktoken으로 토큰 수 계산

`tiktoken`으로 다양한 텍스트의 토큰 수를 계산하고 문자 수와 비교하세요.

In [ ]:
# TODO: 여기에 코드를 작성하세요
# 힌트 1: enc = tiktoken.get_encoding("cl100k_base") 로 인코더 생성
# 힌트 2: tokens = enc.encode(text) 로 토큰화
# 힌트 3: len(tokens) 로 토큰 수 확인

# 인코더 생성
# enc = ...

texts = {
    "영어": "Hello, world!",
    "한글": "안녕하세요, 세상!",
    "코드": "def fibonacci(n):\n    if n <= 1:\n        return n\n    return fibonacci(n-1) + fibonacci(n-2)",
}

for label, text in texts.items():
    # TODO: 각 텍스트의 토큰 수 계산
    char_count = len(text)
    token_count = 0  # TODO: 실제 토큰 수로 대체
    print(f"{label}: {char_count}자 → {token_count}토큰")

In [ ]:
# 검증 셀
enc = tiktoken.get_encoding("cl100k_base")
english_tokens = len(enc.encode("Hello, world!"))
korean_tokens = len(enc.encode("안녕하세요, 세상!"))

assert english_tokens < len("Hello, world!"), "영어 토큰은 문자 수보다 적어야 합니다"
assert korean_tokens > len("안녕하세요, 세상!") // 2, "한글 토큰은 문자 수와 비슷하거나 더 많아야 합니다"
print(f"영어: {english_tokens}토큰, 한글: {korean_tokens}토큰")
print("✅ 실습 1 완료! 토큰 수 계산이 올바르게 동작합니다.")

## 실습 2: TokenBudget 클래스 구현

토큰 예산을 관리하는 `TokenBudget` 클래스를 구현하세요:
- `__init__(self, model="cl100k_base", max_tokens=8000)`: 초기화
- `count(self, text: str) -> int`: 텍스트 토큰 수 반환
- `remaining` property: 남은 토큰 예산
- `used` property: 사용한 토큰 수
- `consume(self, text: str) -> str | None`: 예산 내 소비 (초과 시 None 반환)
- `truncate_to_budget(self, text: str, reserve: int = 100) -> str`: 예산에 맞게 절삭

In [ ]:
# TODO: 여기에 코드를 작성하세요
# 힌트 1: self._enc = tiktoken.get_encoding(model) 으로 인코더 저장
# 힌트 2: consume()은 토큰 수 <= remaining이면 _used 증가 후 text 반환
# 힌트 3: truncate_to_budget()은 available = remaining - reserve 만큼 토큰을 decode

class TokenBudget:
    """토큰 예산 관리.
    
    LLM에 보낼 프롬프트의 토큰 수를 정확하게 관리합니다.
    """
    
    def __init__(self, model: str = "cl100k_base", max_tokens: int = 8000):
        # TODO: 초기화 구현
        pass
    
    def count(self, text: str) -> int:
        """텍스트의 토큰 수를 반환합니다."""
        # TODO: 구현
        pass
    
    @property
    def remaining(self) -> int:
        """남은 토큰 예산."""
        # TODO: 구현
        pass
    
    @property
    def used(self) -> int:
        """사용한 토큰 수."""
        # TODO: 구현
        pass
    
    def consume(self, text: str) -> str | None:
        """예산 내에서 텍스트를 소비합니다.
        
        Returns:
            예산 내이면 원본 텍스트, 초과 시 None
        """
        # TODO: 구현
        pass
    
    def truncate_to_budget(self, text: str, reserve: int = 100) -> str:
        """남은 예산에 맞게 텍스트를 토큰 단위로 절삭합니다."""
        # TODO: 구현
        # available = self.remaining - reserve
        # tokens = self._enc.encode(text)
        # if len(tokens) <= available: 전체 반환
        # else: tokens[:available]만 decode하여 반환 ("\n... [토큰 한도로 절삭됨]" 추가)
        pass

In [ ]:
# 검증 셀 - 기본 동작
budget = TokenBudget(max_tokens=1000)
assert budget.remaining == 1000, f"초기 remaining이 1000이어야 합니다. 현재: {budget.remaining}"
assert budget.used == 0, f"초기 used가 0이어야 합니다. 현재: {budget.used}"

text = "안녕하세요, 이것은 테스트입니다."
token_count = budget.count(text)
assert token_count > 0, "토큰 수가 0보다 커야 합니다"

result = budget.consume(text)
assert result == text, "consume() 성공 시 원본 텍스트를 반환해야 합니다"
assert budget.used == token_count, f"used가 {token_count}여야 합니다. 현재: {budget.used}"
assert budget.remaining == 1000 - token_count

print(f"토큰 수: {token_count}, 사용: {budget.used}, 남은: {budget.remaining}")
print("✅ 실습 2 완료! TokenBudget 기본 동작이 올바릅니다.")

## 실습 3: TokenBudget 실전 활용

총 500 토큰 예산으로 프롬프트를 구성하세요:
1. 시스템 프롬프트 소비
2. 사용자 질문 소비
3. 대용량 코드를 예산에 맞게 절삭

In [ ]:
# TODO: 여기에 코드를 작성하세요
# 힌트 1: TokenBudget(max_tokens=500) 으로 작은 예산 설정
# 힌트 2: consume()으로 시스템 프롬프트와 질문을 소비
# 힌트 3: truncate_to_budget()으로 대용량 코드를 절삭

# 대용량 코드 샘플 (실제는 수천 토큰이지만 여기서는 적당한 크기)
large_code = '''
class DataProcessor:
    """데이터를 처리하는 클래스."""
    
    def __init__(self, config: dict):
        self.config = config
        self.cache = {}
    
    def process(self, data: list) -> list:
        """데이터를 처리합니다."""
        results = []
        for item in data:
            validated = self._validate(item)
            if validated:
                transformed = self._transform(validated)
                results.append(transformed)
        return results
    
    def _validate(self, item: dict) -> dict | None:
        if "id" not in item:
            return None
        return item
    
    def _transform(self, item: dict) -> dict:
        return {"id": item["id"], "value": item.get("value", 0) * 2}
''' * 3  # 3배 반복으로 크기 증가

# TODO: budget을 만들고 프롬프트를 구성하세요
budget = None  # TokenBudget(max_tokens=500)
system_prompt = "당신은 코드 분석 전문가입니다. 다음 코드를 분석하세요."
user_query = "이 코드에서 버그를 찾아주세요."

# TODO: consume()으로 시스템 프롬프트와 질문 소비
# TODO: truncate_to_budget()으로 코드 절삭
truncated_code = None

print(f"원본 코드 토큰: (계산 필요)")
print(f"절삭 후 코드 토큰: (계산 필요)")

In [ ]:
# 검증 셀
assert budget is not None, "TokenBudget을 생성하세요"
assert truncated_code is not None, "truncate_to_budget()을 호출하세요"

budget_check = TokenBudget(max_tokens=500)
original_tokens = budget_check.count(large_code)
truncated_tokens = budget_check.count(truncated_code)

assert truncated_tokens < original_tokens, "절삭 후 토큰이 줄어야 합니다"
print(f"원본 코드 토큰: {original_tokens}")
print(f"절삭 후 코드 토큰: {truncated_tokens}")
print(f"절감률: {(1 - truncated_tokens/original_tokens)*100:.0f}%")
print("✅ 실습 3 완료! TokenBudget으로 프롬프트를 효율적으로 관리했습니다.")

## 정리

이번 실습에서 배운 내용:
1. `tiktoken`으로 정확한 토큰 수를 계산할 수 있다
2. 문자 수 기반 절삭은 부정확하고, 토큰 기반 절삭이 필요하다
3. `TokenBudget`으로 LLM 프롬프트를 예산 내에서 최적화할 수 있다

## 다음 모듈
- **실습 2**: `02_annotated_reducer.ipynb` - LangGraph Annotated Reducer 패턴